In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import torch


model_by = "Qwen"
model_tag = "Qwen2.5-7B-Instruct"

# model_by = "bartowski"
# model_tag = "Qwen2.5-3B-Instruct-GGUF"

model_name = model_by+"/"+model_tag
folder_path = os.path.join(os.path.expanduser("~"), "LLM-Model")
model_dir = folder_path+"/"+model_tag


# Global Configuration
DEVICE_CONFIG = "cpu"  # "auto", "cpu", or "mps"
DEVICE_CONFIG = "mps"  # "auto", "cpu", or "mps"
TORCH_DTYPE = "auto"   # "auto" or specific type


def get_model_and_tokenizer():
    """Load model/tokenizer with proper settings"""

    if not os.path.exists(model_dir):
        print("Downloading and saving model locally...")
        os.makedirs(model_dir, exist_ok=True)

        # Load with trust_remote_code for custom components
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=TORCH_DTYPE,
            device_map=DEVICE_CONFIG,
            trust_remote_code=True
        )
        model.save_pretrained(model_dir)

        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )
        tokenizer.save_pretrained(model_dir)
    else:
        print("Loading local model from", model_dir)
        model = AutoModelForCausalLM.from_pretrained(
            model_dir,
            torch_dtype=TORCH_DTYPE if TORCH_DTYPE != "auto" else None,
            device_map=DEVICE_CONFIG,
            trust_remote_code=True
        )

        if DEVICE_CONFIG == "mps":
            model = model.to(torch.device("mps"))

        # Load tokenizer with trust_remote_code
        tokenizer = AutoTokenizer.from_pretrained(
            model_dir,
            trust_remote_code=True
        )

    return model, tokenizer


# Initialize once and reuse
model, tokenizer = get_model_and_tokenizer()

model-00002-of-00004.safetensors:  79%|#######9  | 3.07G/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [16]:
import requests
import json
from bs4 import BeautifulSoup
import re


def fetch_webpage_content(url, tag=None, element_id=None):
    """Fetch HTML content of specific element or entire page"""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        target_element = None

        # Element selection priority
        if element_id:
            target_element = soup.find(id=element_id)
        elif tag:
            target_element = soup.find(tag)
        else:
            target_element = soup.find("body")

        # Get outer HTML including children and attributes
        if target_element:
            content = str(target_element)
        else:
            content = str(soup)  # Fallback to full page HTML

        print(f"Fetched URL: {url}")
        print(f"Content length: {len(content)}")
        print(f"Content sample: {content[:15000]}")
        return content  # [:15000]  # Conservative truncation for model context

    except Exception as e:
        print(f"Error fetching content: {e}")
        return None


def generate_response(messages):
    """Generate response using the model with chat template"""
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1024,
        temperature=0.7
    )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


def parse_json(json_str):
    """Parse and validate JSON output with error recovery"""
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        # Try to extract JSON from markdown code blocks
        json_match = re.search(r'```json\n(.*?)\n```', json_str, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group(1))
            except json.JSONDecodeError:
                pass
        # Try to find the first valid JSON structure
        json_match = re.search(r'(\{.*\}|\[.*\])', json_str, re.DOTALL)
        if json_match:
            try:
                return json.loads(json_match.group(0))
            except json.JSONDecodeError:
                pass
        return None


def news_pipeline(url, prompt):
    """Main pipeline to process news from URL to structured JSON"""
    # Step 1: Fetch webpage content
    content = fetch_webpage_content(url)  # , tag="main"
    if not content:
        return {"error": "Failed to fetch webpage content"}

    # Step 2: Extract news list
    extraction_messages = [
        {"role": "system", "content": "You are html scraper."},
        # Truncate to avoid token limits
        {"role": "user",
            "content": f"{prompt}\n\nWebsite Content:\n{content[:15000]}"}
    ]
    news_list = generate_response(extraction_messages)

    print(f"news_list: {news_list}")

    # Step 3: Convert to JSON
    json_prompt = """Convert this news list into a JSON array of objects with:
    - title (string)
    - summary (string)
    - date (string in YYYY-MM-DD format)
    - url (string if available)
    
    News List:
    """
    conversion_messages = [
        {"role": "system", "content": "You are a JSON formatting assistant. Return only valid JSON."},
        {"role": "user", "content": f"{json_prompt}\n{news_list}"}
    ]
    json_output = generate_response(conversion_messages)

    # Step 4: Parse and validate JSON
    parsed = parse_json(json_output)
    return parsed if parsed else {"error": "Invalid JSON format", "raw_output": json_output}


# Example usage
if __name__ == "__main__":
    result = news_pipeline(
        url="https://www.bbc.com/news",
        # prompt="List the all news headlines with their summaries and dates and link URLs and if available news image urls"
        prompt="List the top 5 current news headlines with their summaries"
    )
    print(json.dumps(result, indent=2))

Fetched URL: https://www.bbc.com/news
Content length: 254587
Content sample: <body><div id="__next"><div class="app"><noscript><style>.hide-when-no-script { visibility: hidden; }</style></noscript><div class="sc-3192fecd-0 ifUzfO" data-testid="backdrop"></div><div class="sc-db60fb3f-0 hyYukw"><a aria-label="Skip to content" class="sc-db60fb3f-1 bfyvlX" href="#main-content" role="link">Skip to content</a></div><div class="sc-eb24677f-0 cxUgbQ" data-component="ad-slot" data-testid="ad-unit"><div class="dotcom-ad" data-testid="dotcom-interstitial" id="dotcom-interstitial"></div></div><header class="sc-d6c9e5b8-0 dBUlMZ"><div class="sc-d6c9e5b8-1 eCirE"><div class="sc-d6c9e5b8-5 cYRsBO hide-when-no-script"><button aria-expanded="false" aria-label="Open menu" class="sc-d6c9e5b8-3 eTbKet" role="button"><svg aria-hidden="true" category="actions" class="sc-735e4804-0 cCvcou" height="20" icon="list-view-text" viewbox="0 0 32 32" width="20"><path d="M1 7.5h30V1.9H1v5.6zm0 22.6h30v-5.6H1v5.6zm0-1